# Mnemo on Ego4D: Experiment D (Natural Language Queries)

Real first-person videos from many places (8-minute clips) and **questions written by people**
(*"Where did I put the drill?"*), each with the moment where the answer can be seen. We measure:
1. **Trigger:** of the moments people asked about, how many did our trigger send to the big model (vs every N-th window)?
2. **Diary:** does the diary have that moment?
3. **Answer:** does Mnemo point to the right moment (vs asking the big model directly)?

**Settings (right panel):** Accelerator **GPU T4** · Internet **On** · Input: dataset **mnemofyp** (the NEW version with `evaluation/ego4d_nlq.py`)
· **Add-ons → Secrets**: tick `Access Key ID`, `Secret Access Key`, `Region`.

**First time:** keep `QUICK = True` (3 clips, about 1 hour) and use **Run All** to check everything works.
**Then:** set `QUICK = False` → **Save Version → Save & Run All (Commit)**, about 7 hours; you can close the browser.
Download `ego4d_results.zip` from the finished version's **Output**.

The Ego4D videos are kept in `/tmp` and are never saved to the output (the licence does not allow sharing them).

In [ ]:
QUICK = True          # True: 3 clips to check everything (~1 h). False: the full test (~7 h).
N_FULL, N_EXTRA = (3, 0) if QUICK else (20, 20)   # clips through the big model / extra clips for the trigger-only test
SKIP = 0 if QUICK else 3   # the 3 quick-run clips were used to fix the search, so the real test leaves them out
SEED = 0

## 1. Code + packages (~3 min)

In [ ]:
import os, shutil, glob, sys, subprocess, json, time
WORK = '/kaggle/working'
CODE = f'{WORK}/mnemo'
code_dir = code_zip = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'mnemo.py' in files and 'modules' in dirs:
        code_dir = root
    if 'mnemo_code.zip' in files:
        code_zip = os.path.join(root, 'mnemo_code.zip')
print('code folder:', code_dir, '| zip:', code_zip)
assert code_dir or code_zip, 'Code not found: add the mnemofyp dataset (Add Input) and wait until it is attached'
shutil.rmtree(CODE, ignore_errors=True)
if code_dir:
    shutil.copytree(code_dir, CODE)
else:
    os.makedirs(CODE)
    !cd "{CODE}" && unzip -q "{code_zip}"
%cd {CODE}
assert os.path.exists('evaluation/ego4d_nlq.py'), 'OLD code: upload the new mnemo_code.zip as a new version of mnemofyp'
!pip install -q -U "transformers>=4.57" accelerate mediapipe ultralytics ego4d
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch Accelerator to GPU T4!')

## 2. Ego4D access + questions (~1 min)
The keys come from Kaggle Secrets and are written to `~/.aws` (not to the output).

In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
aws = os.path.expanduser('~/.aws')
os.makedirs(aws, exist_ok=True)
with open(f'{aws}/credentials', 'w') as f:
    f.write('[default]\n'
            f'aws_access_key_id = {s.get_secret("Access Key ID")}\n'
            f'aws_secret_access_key = {s.get_secret("Secret Access Key")}\n')
with open(f'{aws}/config', 'w') as f:
    f.write(f'[default]\nregion = {s.get_secret("Region")}\n')
os.chmod(f'{aws}/credentials', 0o600)

E4 = '/tmp/ego4d'                 # NOT in /kaggle/working: videos must not end up in the output
!ego4d --output_directory {E4} --datasets annotations --benchmarks EM -y > /tmp/ego4d_annotations.log 2>&1
NLQ_FILE = glob.glob(f'{E4}/**/nlq_val.json', recursive=True)[0]
print('questions file:', NLQ_FILE)

## 3. Download the test clips (~72 MB each, a few minutes)
One clip per source video, so the clips come from different people and places.

In [ ]:
!python -m tools.fetch_ego4d --nlq "{NLQ_FILE}" --out {E4} --clips {N_FULL} --extra {N_EXTRA} --seed {SEED} --skip {SKIP}
SEL = json.load(open(f'{E4}/selected.json'))
FULL = [c for c in SEL if c['full']]
EXTRA = [c for c in SEL if not c['full']]
print(len(FULL), 'full clips,', len(EXTRA), 'trigger-only clips')

## 4. Load the models (~5 min)

In [ ]:
import config as C
from modules.vlm_logger import VLMDescriber
from modules.hoi_gate import HOIGate
t = time.time()
describer = VLMDescriber(C.VLM_MODEL)
gate = HOIGate('models/hoi_tcn.pt')
print(f'loaded in {time.time() - t:.0f}s | prompt {C.VLM_PROMPT_VERSION} | trigger threshold {C.GATE_THRESHOLD}')

## 5. RESULTS: Experiment D
Each full clip: trigger + big model on every 4-s window (cached), then the diary is rebuilt for each system,
and every question is answered by pointing to one diary line. Finished clips are cached:
if Kaggle stops, run this cell again and it continues.

In [ ]:
from evaluation.ego4d_nlq import load_nlq, run_nlq
from IPython.display import Markdown, display
RESULTS = f'{WORK}/ego4d_results_seed{SEED}' + ('_quick' if QUICK else '')
nlq = load_nlq(NLQ_FILE)
results, summary = run_nlq(FULL, nlq, describer, gate, RESULTS, trigger_only=EXTRA)
display(Markdown(summary))

## 6. Save
Download `ego4d_results.zip` (tables, every answer, the diaries and cached model text; no video).

In [ ]:
shutil.make_archive(f'{WORK}/ego4d_results', 'zip', RESULTS)
print(open(f'{RESULTS}/ego4d_nlq.md').read())
os.chdir(WORK)
from IPython.display import FileLink
display(FileLink('ego4d_results.zip'))